# 실습 ③ 다중 벡터 검색기 + 최종 체인 — 연습본


색인 카드(요약문)를 **서랍**에 꽂고, 원본을 **서가**에 보관해 연결합니다. 오늘 수업의 마지막 조각입니다.

---

## 오늘 만드는 것

```
질문 ──▶ 요약으로 검색 ──▶ 원본 꺼내기 ──▶ 멀티모달 LLM ──▶ 답변
         (색인 카드)       (진짜 책)        (눈 달린 모델)
```

## 진행 방법

1. 셀을 **위에서부터 순서대로** 실행합니다 (`Shift + Enter`)
2. 코드 셀의 **주석을 따라 코드를 직접 작성**합니다 (`# TODO` 는 핵심 포인트)
3. 각 단계의 **확인 포인트**를 통과하면 다음으로 넘어갑니다

## ⚠️ 마지막 8번 셀을 꼭 하세요

**Before/After 비교**는 과제 채점에서 7점이고, 포트폴리오에서 가장 강력한 자료입니다. 시간이 모자라면 9번(번역 체인)을 건너뛰고 8번을 하세요.

## 0. 결과 복원

지금까지 만든 모든 재료를 한 번에 불러옵니다.

> ⚠️ `FileNotFoundError`가 나면 앞 교시의 저장 셀을 실행하지 않은 것입니다.

In [10]:
# ── 0. 결과 복원 ──────────────────────────────────────────
# 1) 필요한 모듈 import: os, re, time, uuid, pickle, pathlib.Path
import os #운영체제 기능(환경변수, 파일, 폴더 다루기 위한 기능)
import re #정규표현식으로 문자열패턴, 만든 검사,검색,치환
import time #현재 시각 확인, 시간측정, 대기 (체인 답변 생성시간 측정)
import uuid # 겹치지 않는 고유 ID 생성 uuid.uuid4() 요약본, 원본을 doc_id로 페어링
import pickle #파이썬 객체를 
from pathlib import Path
# 2) python-dotenv 의 load_dotenv import
from dotenv import load_dotenv

# 3) .env 가 있는 폴더를 프로젝트 루트로 자동 탐색
#    - Path.cwd() 와 그 상위 폴더(parents)를 차례로 확인해 (p / ".env") 가 존재하는 첫 경로를 PROJECT_ROOT 로 지정
#    - 찾지 못하면 Path.cwd() 를 사용
#    - PROJECT_ROOT 출력 → c:\workspaces\multimodal-rag 이면 정상
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if(p / ".env").exists() ),Path.cwd(),
)
print("프로젝트 루트: ", PROJECT_ROOT)

# 4) PROJECT_ROOT 의 .env 파일 로드 (load_dotenv)
load_dotenv(PROJECT_ROOT / ".env")

# 5) OPENAI_API_KEY 가 없으면 ValueError 발생시키기
try:
    openai_api_key = os.environ["OPENAI_API_KEY"]
    if not openai_api_key.strip():
        raise ValueError("OPENAI_API_KEY 값이 비어있습니다.")
except KeyError:
    raise ValueError("OPENAI_API_KEY 키가 없습니다. .env파일을 확인해주세요")

# 6) processed 경로 = PROJECT_ROOT / "data" / "processed"
processed = PROJECT_ROOT / "data" / "processed"

# 7) 추출 결과 복원: processed/"extracted.pkl" 을 pickle 로 읽기 ("rb")
#    - texts, tables, texts_2k_token 꺼내기
with open (processed/"extracted.pkl", "rb") as f:
    doc1 = pickle.load(f)
    texts,tables, texts_2k_token= doc1["texts"],doc1["tables"], doc1["texts_2k_token"]


# 8) 요약 결과 복원: processed/"summaries.pkl" 을 pickle 로 읽기
#    - text_summaries, table_summaries, image_summaries, img_base64_list 꺼내기
with open (processed/"summaries.pkl", "rb") as f:
    doc1 = pickle.load(f)
    text_summaries, table_summaries, image_summaries, img_base64_list= doc1["text_summaries"],doc1["table_summaries"], doc1["image_summaries"],doc1["img_base64_list"]

# 9) "복원 완료" 와 함께 텍스트·표·이미지의 원본 수 / 요약 수 출력
print("복원 완료")
print(f"텍스트 원본 {len(texts_2k_token):3d}건 / 요약 {len(text_summaries)}건")
print(f"표 원본 {len(tables):3d}건 / 요약 {len(table_summaries)}건")
print(f"이미지 원본 {len(img_base64_list):3d}건 / 요약 {len(image_summaries)}건")




프로젝트 루트:  c:\workspaces\multimodal-rag
복원 완료
텍스트 원본  27건 / 요약 27건
표 원본  35건 / 요약 35건
이미지 원본   6건 / 요약 6건


> ✅ **확인 포인트:** 텍스트·표·이미지의 원본 수와 요약 수가 각각 같아야 합니다. (표는 0건일 수 있습니다)

## 1. 다중 벡터 검색기의 구조

> **비유:** **도서관의 색인 카드**입니다. 사서는 책을 다 읽고 찾지 않습니다. 요약이 적힌 **색인 카드로 검색**하고, 찾으면 **서가에서 진짜 책을 꺼내** 건네줍니다.

| 구성 요소 | 비유 | 들어가는 것 | 역할 |
| --- | --- | --- | --- |
| `vectorstore` (Chroma) | 색인 카드 서랍 | 요약문 임베딩 | **검색** |
| `docstore` (InMemoryStore) | 서가 | 원본 텍스트·표·Base64 이미지 | **전달** |
| `id_key` | 청구기호 | `doc_id` | 둘을 **연결** |

오늘 배우는 것의 핵심은 한 줄입니다.

> **요약과 원본에 같은 번호표를 붙인다.**

## 2. 검색기 만들기

아래 셀의 주석을 따라 코드를 작성하세요.

> `InMemoryStore`는 메모리에만 저장됩니다. 커널을 재시작하면 사라지므로 실제 서비스에서는 파일이나 Redis를 씁니다.

In [11]:
# ── 2. 검색기 만들기 ──────────────────────────────────────
# langchain 1.x 에서 두 클래스의 위치가 바뀌었습니다.
# 어느 버전을 설치했든 동작하도록 두 경로를 모두 시도합니다. (try / except ImportError)

# 1) MultiVectorRetriever import
#    - langchain 1.x : langchain_classic.retrievers.multi_vector
#    - langchain 0.3 : langchain.retrievers.multi_vector
try:
    from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
except ImportError:
    print("MultiVectorRetriever 클래스가 필요합니다")
    
# 2) InMemoryStore import
#    - langchain 1.x : langchain_core.stores
#    - langchain 0.3 : langchain.storage
try:
    from langchain_core.stores import InMemoryStore
except ImportError:
    print("InMemoryStore 클래스가 필요합니다")

# 3) Chroma(langchain_chroma), OpenAIEmbeddings(langchain_openai) import
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 4) 요약문이 들어갈 벡터 DB (= 색인 카드 서랍)
#    - collection_name="multimodal_rag"
#    - embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
     collection_name="multimodal_rag" , embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")
)
# 셀을 다시 실행해도 이전 요약이 남지 않도록 컬렉션을 비운다.
# (안 비우면 요약은 검색되는데 새 docstore엔 원본이 없어서 검색 결과가 빈 리스트가 됨)
vectorstore.reset_collection()

# 5) 원본이 들어갈 저장소 (= 서가) : InMemoryStore()
store = InMemoryStore()

# 6) 연결 키(= 청구기호) : id_key = "doc_id"
id_key = "doc_id"
# 7) MultiVectorRetriever(vectorstore=..., docstore=..., id_key=...) 생성
retriever = MultiVectorRetriever(
     vectorstore= vectorstore, docstore= store, id_key = id_key
)


# 8) "검색기 생성 완료" 출력
print("검색기 생성 완료")

검색기 생성 완료


## 3. 요약과 원본을 같은 ID로 짝짓기

**TODO 1~2.** 주석을 따라 함수를 작성하세요.

💡 **힌트:** 위 표를 보세요.
- 요약문은 **검색**에 쓰입니다 → 어디에 넣어야 할까요?
- 원본은 **전달**에 쓰입니다 → 어디에 넣어야 할까요?

> ⚠️ **이 셀을 두 번 실행하면 데이터가 중복됩니다.** 다시 돌리려면 2번 셀부터 재실행하세요.

In [12]:
from langchain_core.documents import Document as _Doc


def add_documents(retriever, doc_summaries, doc_contents):
    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
    doc_ids = [str(uuid.uuid4()) for _ in doc_contents]

    summary_docs = [
        _Doc(page_content=s, metadata={id_key: doc_ids[i]})
        for i, s in enumerate(doc_summaries)
    ]

    # TODO 1: 요약본은 어디에 넣을까요? (vectorstore / docstore)
    retriever.vectorstore.add_documents(summary_docs)

    # TODO 2: 원본은 어디에 넣을까요?
    retriever.docstore.mset(list(zip(doc_ids, doc_contents)))


if text_summaries:
    add_documents(retriever, text_summaries, texts_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)

print("색인 완료")
print("  벡터 DB에 저장된 요약 수:", len(vectorstore.get()["ids"]))

색인 완료
  벡터 DB에 저장된 요약 수: 68


## 4. 검색 테스트 — 요약으로 찾고 원본을 받는다

**TODO 3.** 본인 PDF에 맞는 질문을 적으세요.

💡 **힌트:** **그래프나 표를 봐야만** 답할 수 있는 질문이어야 합니다. 텍스트에도 똑같이 적혀 있는 내용을 물으면 나중에 Before/After 차이가 안 납니다.

> 좋은 예: "그래프에 나타난 2023년 추세를 설명해줘"
> 나쁜 예: "이 문서는 무엇에 관한 것인가요?"

In [13]:
# ── 4. 검색 테스트 ────────────────────────────────────────
# TODO 3: 그래프나 표를 봐야만 답할 수 있는 질문을 query 에 적으세요
query = "과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?"
# 1) retriever.invoke(query) 결과를 found 에 저장
# 2) 검색된 원본 수 출력
found = retriever.invoke(query)
# 1. 질문을 임베딩으로 바꾼다.
# 2. vectorstore에서 질문과 가장 비슷한 요약문을 찾는다.(Chroma)
# 3. 찾은 요약의 metadata["doc_id"] 읽는다.
# 4. doc_id(번호표) docstore 에서 원본을 꺼내 반환한다. (Inmemory Store)

print("검색된 원본 수:", len(found))
print()

# 3) found 를 순회하며
#    - content = doc.page_content (Document 가 아니면 doc 그대로)
#    - 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 에 맞으면 "이미지(Base64)", 아니면 "텍스트/표"
#    - 종류, 길이, 앞 120자(줄바꿈 제거) 출력

for i, doc in enumerate(found):
    content = doc.page_content if hasattr(doc, "page_content") else doc
    is_image = bool(re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content))
    #Base64에 쓰는 문자만 100개이상 이어지고, 끝에 = 이 0~2개 붙은 문자열은 이미지이다.
    kind = "이미지(Base64)" if is_image else "텍스트/표"
    print(f"[{i}] {kind} — 길이 {len(content):,}자")
    print("   ", content[:120].replace("\n", " "), "...")
    print()

검색된 원본 수: 4

[0] 이미지(Base64) — 길이 142,416자
    /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/ ...

[1] 텍스트/표 — 길이 971자
    주1) 위의 표와 그래프는 분석시작일인 [2003년 01월 02일]부터 [2023년 11월 02일]까지의 해당 상품의 일 별 수익률 변화를 분석한 자료입니다.  주2) 분석시작일부터 [2023년 11월 02일]까지는 ...

[2] 텍스트/표 — 길이 491자
    구 분 내 용 ① 1차 자동조기상환평가일에 기초자산의 자동조기상환평가 가격이 최초기준가격의 80% 이상인 경우 ② 2차 자동조기상환평가일에 기초자산의 자동조기상환평가 가격이 최초기준가격의 80% 이상인 경우 자동 ③ ...

[3] 텍스트/표 — 길이 224자
    차수 자동조기상환 발생조건 1차 기초자산의 1차 자동조기상환평가가격이 최초기준가격의 80%이상인 경우 2차 기초자산의 2차 자동조기상환평가가격이 최초기준가격의 80%이상인 경우 3차 기초자산의 3차 자동조기상환평가가 ...



> ✅ **확인 포인트:** 검색 결과에 **이미지(Base64)가 최소 1건** 들어 있어야 합니다.
>
> 하나도 없다면 이미지 요약의 품질이 낮아 검색이 안 되는 것입니다. 6교시로 돌아가 프롬프트를 보강하세요.

> 💡 **여기가 오늘의 핵심 장면입니다.** 검색어와 글자가 하나도 안 겹치는 원본이 나왔죠? 요약문을 거쳐서 찾았기 때문입니다.

## 5. 검색 결과를 텍스트와 이미지로 분리

멀티모달 LLM에 넘기려면 둘을 다른 형식으로 포장해야 합니다. Base64 문자열의 **생김새**로 구분합니다.

아래 셀의 주석을 따라 코드를 작성하세요.

In [14]:
# ── 5. 검색 결과를 텍스트와 이미지로 분리 ─────────────────
# 1) langchain_core.documents 의 Document import
from langchain_core.documents import Document

# 2) split_image_text_types(docs) 함수 정의
#    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
#    - b64_images, texts_out 빈 리스트 준비
#    - Document 면 page_content 를 꺼내 문자열로 만들기
#    - Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
#      → 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 매칭 시 이미지, 아니면 텍스트
#    - {"images": b64_images, "texts": texts_out} 반환
def split_image_text_types(docs):
    b64_images=[]
    texts_out = []

    for doc in docs:
        doc.page_content if isinstance(doc,Document) else doc
        if re.match("^[A-Za-z0-9+/]{100,}={0,2}$",doc):
            b64_images.append(doc)
        else:
            texts_out.append(doc)
    return {"images":b64_images, "texts" :texts_out}

# 3) 앞서 검색한 결과(found)로 동작 확인 → 이미지 / 텍스트 건수 출력
split_data = split_image_text_types(found)
print("이미지: ",len(split_data["images"]),"건")
print("표:", len(split_data["texts"]),"건")

이미지:  1 건
표: 3 건


## 6. 최종 멀티모달 체인 만들기

체인의 구조는 세 칸입니다.

```
{검색 + 질문}  →  img_prompt_func  →  멀티모달 LLM  →  StrOutputParser
   재료 모으기      메시지로 조립        답변 생성        문자열로 꺼내기
```

**TODO 4~5.** 주석을 따라 체인을 작성하세요.

💡 **힌트**
- TODO 4: 6교시에서 Base64 앞에 붙였던 것과 같은 접두사입니다
- TODO 5: 위 그림의 3번 칸에 들어갈 변수 이름입니다

In [15]:
# ── 6. 최종 멀티모달 체인 만들기 ──────────────────────────
# 1) import
#    - HumanMessage (langchain_core.messages)
#    - StrOutputParser (langchain_core.output_parsers)
#    - ChatPromptTemplate (langchain_core.prompts)
#    - RunnableLambda, RunnablePassthrough (langchain_core.runnables)
#    - ChatOpenAI (langchain_openai)
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda , RunnablePassthrough
from langchain_openai import ChatOpenAI

# 2) img_prompt_func(data_dict) 함수 정의
#    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
#    - messages 빈 리스트 준비
#    - 1) 이미지를 먼저 넣습니다 (모델이 그림을 먼저 보게)
#         data_dict["context"]["images"] 마다 {"type": "image_url", "image_url": {"url": ...}} 추가
#         TODO 4: Base64 앞에 붙일 접두사를 완성하세요 (data:image/jpeg;base64,)
#    - 2) 텍스트 컨텍스트와 질문을 넣습니다
#         texts 를 줄바꿈으로 합치고 {"type": "text", "text": ...} 추가
#         지시문: "아래 자료를 근거로 질문에 한국어로 답하세요. 자료에 없는 내용은 모른다고 답하세요."
#         + 질문(data_dict["question"]) + 참고 텍스트
#    - [HumanMessage(content=messages)] 반환
def img_prompt_func(data_dict):
    messages = []

    # 1. 이미지를 먼저 넣는다.
    for image in data_dict["context"]["images"]:
        messages.append(
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image}"}}
        )
    # 2. 텍스트 컨텍스트와 질문을 넣는다.
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages.append({
        "type":"text",
        "text":( "자료를 근거로 질문에 한국어로 답하세요\n"
                 "자료에 없는 내용은 모른다고 답하세요\n\n"
                 f"질문: {data_dict['question']}\n\n"
                 f"참고텍스트: \n{formatted_texts}"
                )
    })
    return [HumanMessage(content=messages)]


# 3) 모델: ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
model = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)

# 4) 체인 조립 (LCEL)
#    {"context": retriever | RunnableLambda(split_image_text_types),
#     "question": RunnablePassthrough()}
#    | RunnableLambda(img_prompt_func)
#    | TODO 5: 멀티모달 LLM 변수
#    | StrOutputParser()
chain=(
    {
        "context": retriever | RunnableLambda (split_image_text_types),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(img_prompt_func)  
    |model
    | StrOutputParser()
)

# 5) "최종 체인 조립 완료" 출력
print("최종 체인 조립 완료")

최종 체인 조립 완료


## 7. 질의 실행

In [16]:
# ── 7. 질의 실행 ──────────────────────────────────────────
# 1) 시작 시간 기록 (time.time())
start = time.time()

# 2) chain.invoke(query) 결과를 answer 에 저장
answer = chain.invoke(query)

# 3) 소요 시간, 질문(Q), 구분선, 답변 출력
print(f"답변 생성 — {time.time() - start:.1f}초\n")
print("Q:", query)
print("-" * 70)
print(answer)

답변 생성 — 5.4초

Q: 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?
----------------------------------------------------------------------
상환 차수별 횟수와 빈도율은 다음과 같습니다:

- 1차: 5009회 (97.21%)
- 2차: 80회 (1.55%)
- 3차: 64회 (1.24%)
- 4차: 0회 (0.00%)
- 5차: 0회 (0.00%)

1차 상환 비율은 97.21%입니다.


> ✅ **확인 포인트:** 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있어야 합니다.
>
> 두루뭉술한 답이 나왔다면 질문을 더 구체적으로 바꿔보세요.

## 8. Before / After 비교 (가장 중요)

> 📌 **과제 채점 7점짜리 셀입니다. 반드시 하세요.**

같은 질문을 **텍스트 전용 RAG**와 **멀티모달 RAG**에 각각 던져 차이를 증거로 남깁니다.

먼저 비교용 텍스트 전용 RAG를 만듭니다. **이미지와 표를 뺀** 것 말고는 모든 조건이 같습니다.

In [34]:
# ── 8-1. 비교용 텍스트 전용 RAG ────────────────────────────
# 이미지와 표를 뺀 것 말고는 모든 조건이 같아야 합니다.

# 1) 텍스트만 들어간 벡터 DB 생성
#    - Chroma(collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"))
#    - texts_2k_token 을 add_texts 로 저장
#    - as_retriever(search_kwargs={"k": 4}) 로 검색기 생성
text_only_vectorstore = Chroma(
    collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)
text_only_vectorstore.add_texts(texts=texts_2k_token)

text_only_retriever = text_only_vectorstore.as_retriever(search_kwargs={"k":4})

# 2) 프롬프트: ChatPromptTemplate.from_template
#    - 멀티모달 체인과 같은 지시문 + {question} + {context}
text_only_prompt = ChatPromptTemplate.from_template(
    "자료를 근거로 질문에 한국어로 답하세요\n"
    "자료에 없는 내용은 모른다고 답하세요\n\n"
    "질문: {question}\n\n"
    "참고텍스트: \n{context}"
)

# 3) 체인 조립
#    {"context": 검색기 | (문서들의 page_content 를 줄바꿈으로 합치는 함수),
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)
#     "question": RunnablePassthrough()}
#    | 프롬프트 | model | StrOutputParser()
text_only_chain = (
    {
        "context": text_only_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    |text_only_prompt
    |model
    |StrOutputParser()
)

# 4) "비교용 텍스트 전용 RAG 준비 완료" 출력
print("비교용 텍스트 전용 RAG 준비 완료")
print(text_only_chain.invoke(query))

비교용 텍스트 전용 RAG 준비 완료
자료에는 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율에 대한 정보가 포함되어 있지 않습니다. 또한 1차 상환 비율에 대한 정보도 제공되지 않았습니다. 따라서 이 질문에 대한 답변은 알 수 없습니다.


In [37]:
# ── 8-2. Before / After 출력 ───────────────────────────────
# 1) 구분선과 질문 출력
print("="*60)
print("질문: ",query)

# 2) [BEFORE] 텍스트 전용 RAG
#    - text_only_chain.invoke(query) 결과를 before 에 저장 후 출력
print("="*60)
print("[BEFORE] 텍스트 전용 RAG")
before = text_only_chain.invoke(query)
print(before)

# 3) [AFTER] 멀티모달 RAG
#    - chain.invoke(query) 결과를 after 에 저장 후 출력
print("="*60)
print("[AFTER] 멀티모달 RAG")
after = chain.invoke(query)
print(after)

질문:  과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?
[BEFORE] 텍스트 전용 RAG
자료에는 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율에 대한 정보가 포함되어 있지 않습니다. 또한, 1차 상환 비율에 대한 정보도 제공되지 않았습니다. 따라서 이와 관련된 내용은 알 수 없습니다.
[AFTER] 멀티모달 RAG
상환 차수별 횟수와 빈도율은 다음과 같습니다:

- 1차: 5009회 (97.21%)
- 2차: 80회 (1.55%)
- 3차: 64회 (1.24%)
- 4차: 0회 (0.00%)
- 5차: 0회 (0.00%)

1차 상환 비율은 97.21%입니다.


**TODO 6. 두 답변의 차이를 적으세요 (더블클릭해서 편집)**

- **텍스트 전용 RAG의 답변은 어땠나요**:
답: 질문 내용이 텍스트 전용 RAG에 있지 않아 답변이 제대로 되지 않았다.
- **멀티모달 RAG의 답변은 어땠나요**:
답: 질문 내용에 맞게 이미지를 찾아 답변을 해주었다.
- **차이가 생긴 이유는**:
답: 멀티모달 RAG는 텍스트뿐만 아니라 표, 이미지도 임베딩 해 검색하기 때문에 pdf전체 내용을 물어봐도 원활하게 답변이 가능하다.

> 💡 **차이가 잘 안 드러난다면** 질문이 잘못된 것입니다. 텍스트에도 같은 내용이 적혀 있을 가능성이 큽니다. **그래프에만 있는 정보**를 묻는 질문으로 바꿔 4번 셀부터 다시 돌리세요.

> 📸 **지금 이 출력을 스크린샷으로 저장하세요.** 포트폴리오에서 가장 강력한 자료입니다.

# 9. 질문 5개 비교해보기

In [19]:
# ── 질의응답 5건: 질문 / 답변 / 근거가 된 원본 종류 ────────
import re  # 정규표현식: Base64 이미지 문자열인지 판별할 때 사용

# 1) invest.pdf 내용을 보고 만든 질문 5개
#    - 근거가 이미지·표·텍스트로 골고루 나오도록 구성
questions = [
    "과거 20년 시뮬레이션에서 1차 자동조기상환으로 상환된 비율은 몇 %인가요?",      # 이미지(상환 횟수·빈도 그림)
    "2차 자동조기상환평가일은 언제이고, 조건을 충족하면 액면금액의 몇 %를 받나요?",  # 표(차수별 평가일·상환금액)
    "만기평가가격이 최초기준가격의 80% 미만이면 만기에 얼마를 돌려받나요?",          # 표(만기상환금액)
    "중도상환을 신청하면 공정가액 결정과 대금 지급은 각각 언제 이루어지나요?",       # 이미지(일정 흐름도) + 표
    "이 상품에 투자할 때 원금 손실이 생길 수 있는 요인은 무엇인가요?",              # 텍스트(투자위험 본문)
]


# 2) 검색된 원본 1건이 어떤 종류인지 판별하는 함수
def kind(doc):
    # docstore에서 꺼낸 원본은 보통 문자열이지만, Document 객체일 수도 있으니 page_content를 꺼낸다
    c = doc.page_content if hasattr(doc, "page_content") else doc
    # Base64 문자(영문/숫자/+/)만 100자 이상 이어지고 끝에 '='이 0~2개 → 이미지
    if re.match(r"^[A-Za-z0-9+/]{100,}={0,2}$", c.strip()):
        return "이미지"
    # 이미지가 아니면 표 또는 텍스트 (둘 다 문자열이라 여기서는 구분하지 않음)
    return "표/ 텍스트"


# 3) 질문 5개를 차례대로 실행하며 결과 출력
for i, q in enumerate(questions):
    # 3-1) 검색만 따로 실행 → 답변의 근거가 된 원본 목록
    #      (요약으로 검색 → doc_id로 docstore에서 원본을 꺼내 반환)
    found = retriever.invoke(q)

    # 3-2) 원본마다 종류를 판별하고, set으로 중복 제거 후 정렬
    #      예: ["이미지", "표/ 텍스트"]
    kinds = sorted({kind(d) for d in found})

    # 3-3) 멀티모달 체인으로 답변 생성 (검색 → 이미지/텍스트 분리 → LLM → 문자열)
    answer = chain.invoke(q)

    # 3-4) 질문 / 답변 / 원본종류 순서로 출력
    print(f"[{i+1}]번 질문: {q}")
    print("답변:", answer.strip())  # 앞뒤 공백·줄바꿈 제거
    print("원본종류:", ", ".join(kinds) if kinds else "(검색 결과 없음)")
    print("-" * 70)  # 질문 사이 구분선


[1]번 질문: 과거 20년 시뮬레이션에서 1차 자동조기상환으로 상환된 비율은 몇 %인가요?
답변: 자료에 따르면, 1차 자동조기상환으로 상환된 비율에 대한 정보는 제공되지 않습니다. 따라서 해당 비율에 대해서는 알 수 없습니다.
원본종류: 이미지, 표/ 텍스트
----------------------------------------------------------------------
[2]번 질문: 2차 자동조기상환평가일은 언제이고, 조건을 충족하면 액면금액의 몇 %를 받나요?
답변: 2차 자동조기상환평가일은 2027년 11월 12일이며, 조건을 충족하면 액면금액의 106.10%를 받습니다.
원본종류: 표/ 텍스트
----------------------------------------------------------------------
[3]번 질문: 만기평가가격이 최초기준가격의 80% 미만이면 만기에 얼마를 돌려받나요?
답변: 만기평가가격이 최초기준가격의 80% 미만인 경우, 만기에 지급받는 금액은 액면금액의 100.00%입니다.
원본종류: 표/ 텍스트
----------------------------------------------------------------------
[4]번 질문: 중도상환을 신청하면 공정가액 결정과 대금 지급은 각각 언제 이루어지나요?
답변: 중도상환을 신청하면 공정가액 결정은 중도상환 신청일 익거래소영업일에 이루어집니다. 대금 지급은 중도상환 신청일 이후 2영업일에 해당하는 날에 이루어집니다.
원본종류: 표/ 텍스트
----------------------------------------------------------------------
[5]번 질문: 이 상품에 투자할 때 원금 손실이 생길 수 있는 요인은 무엇인가요?
답변: 이 상품에 투자할 때 원금 손실이 생길 수 있는 요인은 다음과 같습니다:

1. **금융투자상품의 특성**: 금융투자상품은 원금 손실이 발생할 수 있으며, 예금자 보호대상이 아닙니다.

2. 

---

## ✅ 체크포인트 4 — 모두 체크되면 완료

- [✅] 질문 하나에 대해 답변이 출력된다
- [✅] 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있다
- [✅] 검색 결과에 이미지(Base64)가 최소 1건 포함된다
- [✅] Before/After 출력을 **캡처해 저장**했다
- [✅] 질의응답 5건 마크다운 표를 복사해 두었다
- [✅] TODO 1~6을 모두 채웠다

## 생각해 볼 질문 (답을 적어보세요)

**Q1. 요약과 원본에 같은 `doc_id`를 붙이지 않으면 어떤 일이 벌어질까요?**

답: 요약은 검색되지만 그 요약에 해당하는 원본을 docstore에서 꺼낼 수 없습니다. 원본을 못 찾기 때문에 결과가 비게 된다.

**Q2. 검색은 요약으로 하는데, 왜 답변 생성에는 원본을 쓸까요? 요약만 쓰면 안 되나요?**

답: 답변은 pdf 전체 내용이 그대로 들어가야하기 때문에, 검색만 요약으로 검색하고, 답변 생성은 요약을 쓴다. 요약을 답변생성에 사용하게 되면 정보의 소실이 있을 수 있다.

**Q3. `InMemoryStore`는 커널을 끄면 사라집니다. 실제 서비스로 만들려면 무엇을 바꿔야 할까요?**

답: 원본과 벡터DB 모두 디스크나 외부 서버에 영구 저장해야 한다.

## 자주 나는 에러

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| `FileNotFoundError: summaries.pkl` |  저장 셀 미실행 | 앞 노트북 마지막 셀 재실행 |
| 검색 결과에 이미지가 없음 | 이미지 요약 품질이 낮음 | 프롬프트 보강 |
| 답변이 "자료에 없습니다" | 검색이 엉뚱한 원본을 가져옴 | 질문을 요약문 표현에 가깝게 수정 |
| 같은 원본이 여러 번 나옴 | `add_documents`를 두 번 실행 | 2번 셀부터 재실행 |
| `RateLimitError` | 질의 5건 연속 호출 | 잠시 대기 후 재실행 |
| Before/After 차이가 없음 | 질문이 텍스트로도 답 가능 | 그래프에만 있는 정보를 묻도록 수정 |
| `NameError: found` | 4번 셀을 안 돌림 | 위에서부터 순서대로 재실행 |

## 과제로 가져갈 것

| 산출물 | 어디서 나왔나 |
| --- | --- |
| Before/After 캡처 | 8번 셀 출력 |
| 질의응답 5건 표 | 10번 셀이 출력한 마크다운 |
| 아키텍처 다이어그램 | 직접 작도 (Excalidraw / Mermaid) |
| `README.md` | 문제 정의 + 위 자료들 |
| `.gitignore` | `.env`, `data/figures/`, `*.pdf` |

